# Pytest

source: [pytest docs](https://docs.pytest.org/en/stable/)

## Install

\> poetry add --group dev pytest

Note that the dev group is just a common group for poetry and by default, if you execute poetry install, dev-dependencies will be installed as well. You've three way to exclude them when installing dependencies in production environment:
- \> poetry install --only main
- \> poetry install --without dev 
- mark the group as optional in pyproject.toml
    ```yaml
    [tool.poetry.group.docs]
    optional = true
    ```
If you make the group optional then to install it in dev environment you should do as follow:
- \> poetry install --with dev 

Note: even though pytest is a tool, we don't install it with pipx but we install it with poetry because pytest needs to import your code to run against it, so it must run in the same venv where code's dependencies are installed 

## Usage

### Basic usage

1. Create the test files (10_pytest_test.py)
2. Execute the following
```bash
    poetry run pytest --version      # To ensure pytest is installed
    poetry run pytest                # To execute tests and show one character per test (.=pass, F=fail, s=skip) 
                                     #    and a single summarized result
    poetry run pytest -v             # To execute tests and show one line per test with name and result
    poetry run pytest --collect-only # To show list of collected tests without running them
```

### Test discovery

pytest will scan current directory and subdirectories to discover and run all files of the form test_*.py or *_test.py, inside these it will collect:
- test prefixed functions outside classes
- test prefixed functions and methods inside Test prefixed classes (just those without the \_\_init\_\_ method)

see [test-discovery](https://docs.pytest.org/en/stable/explanation/goodpractices.html#test-discovery)

### Project Structure

test files are usually placed in a tests directory
```yaml
    pyproject.toml
    src/
        mypkg/
            __init__.py
            app.py
            view.py
    tests/
        test_app.py
        test_view.py
        ...
```

### Test Classes

Tests can be group in test classes with the following naming conventions:
- Test classes's name must have the Test Prefix.
- Test classes must not have the \_\_init\_\_ method
- Test methods's name in the test classes must have the test_ prefix.

Each test method runs in a unique instance of the class.

TestClasses are useful to organize tests and to apply fixtures or parametrization to a whole group of tests at once. 

### Fixtures

Fixtures are functions used to prepare something for the tests (db connection, temp directories, data, ...).

Fixtures syntax:
- Fixture functions must be decorated with @pytest.fixture
- Fixtures can be used inside a test function by adding a parameter with the same name as the fixture
```python
    import pytest

    @pytest.fixture
    def sample_user():
        return {"name": "Ada", "email": "ada@example.com"}

    def test_greeting(sample_user, tmp_path):        # pytest sees the name, calls the fixture
        assert greet(sample_user) == "Hello, Ada"
``` 

Fixtures pros:
- You write the fixture once and then declare the dependency by adding a parameter with the same name to the test functions
- Fixtures defined in a test file are available inside the file
- Fixtures defined in `conftest.py` are available to every test in the same folder or subfolder without the needing to import the file.

Built-in fixtures:
- tmp_path -- creates a temporary path that can be used inside the test
- monkeypatch -- patches attributes and env vars
- capsys -- captures stdout/stderr
- ...

Fixtures scope:
- By default the fixture scope is at function level (every functions gets his own fresh fixture)
- You can modify the scope level with `@pytest.fixture(scope="module")` / class / package / session
- It can be useful when setting up something expensive like a server but be careful because their state will be shared between tests 

Fixtures and Context Managers:
- You can use _yield_ inside fixtures functions to create a context manager as usual and ensuring the resource will be released after the test runs (even if it fails)
- You can add these fixtures to the function parameters as normal fixtures and pytest will be in charge of using the _with_ statement   
```python
    @pytest.fixture
    def db_connection():
        conn = connect("sqlite:///:memory:")
        yield conn
        conn.close()
```

### Marks

Marks are metadata that can be attached to a test function or test class in order to:
- Apply fixtures without injecting result (Mark.usefitxtures)
- Skip a test (Mark.skip or Mark.skipif)
- Run a test against multiple sets of data even with an expected result per set (Mark.parametrize) 
- Expects a test to fail (Mark.xfail)

Mark.usefixtures for setups using side effects:
- Usefixtures can be applied at class level or function level `@pytest.mark.usefixtures("db_connection")` to apply a fixture without injecting its return value
- When applied at class level it's applied automatically once for each test method 
- This is useful to apply side effects (seed a db or monkeypatch envs that will be used inside the test functions)  

Custom marks can also be defined in pyproject.toml in order to tag tests. It can be useful to mark and include or exclude slow tests.

```yaml
    #pyproject.toml
    [tool.pytest.ini_options]
    markers = [
        "slow: tests that take more than a second",
        "integration: requires a real database"
    ]
```
```python
    #test_sample.py
    @pytest.mark.slow
    def test_slow():
        ...
```
```bash
    #bash
    pytest -m slow          # only tagged tests
    pytest -m "not slow"    # everything else
    pytest -m "slow and not integration"
```